# PerioVision AI: stronger landmark models on corrected labels (v3)

**Goal:** lower the periapical bone-loss error and make progression detection more sensitive. Progression is limited by
how fully the landmark model follows a change in the bone crest, so a more accurate model helps both.

**What this notebook does**
1. Builds the DenPAR labels with the **corrected** converter (the crest is always taken from the CEJ's own side of the
   tooth) and leaves out the films duplicated across splits.
2. Trains up to **three** two-site landmark models (CEJ and crest on both sides of each tooth, plus the apex):

   | Member | Model | Image size | Notes |
   |---|---|---|---|
   | `l1280` | YOLO11l-pose (large) | 1280 px | the strongest single model; slowest |
   | `m1280` | YOLO11m-pose | 1280 px | higher resolution than today's model |
   | `m1024` | YOLO11m-pose | 1024 px | same settings as today's model, on clean labels |

3. Scores every finished model **and** their ensemble (all readings averaged per tooth, exactly as the app does), on
   validation and test films, with the same code and reference as every published number.
4. Picks the winner **on validation films only**, then reports its test result and a paired comparison against the
   model deployed today. The app is changed only if that comparison shows a real improvement.

**Before you start:** upload the deployed model `backend/weights/dental_landmark_yolov8n-pose.pt` to
`MyDrive/PerioVision/current/` (replace any older copy) so the notebook can compare against it.

**How to run (it takes several sessions):** `Runtime` → `Change runtime type` → **GPU** (T4 is fine; A100 / L4 is much
faster) → Save, then `Runtime` → **Run all**. Allow Google Drive: pick the same account and **tick every permission
box**. Each session trains the next unfinished model and resumes after a disconnect, so just **Run all** again until
step 8 prints the verdict. On a free T4 expect roughly 3-6 hours per model; with less GPU time, set `RUN` in step 0
to fewer members (any one finished model can already be scored).

**After:** download `MyDrive/PerioVision/export_v3/` to the laptop and ask for it to be installed.

In [ ]:
# 0. Settings
MEMBERS = {
    "l1280": {"model": "yolo11l-pose", "imgsz": 1280, "seed": 0,  "epochs": 250, "patience": 50},
    "m1280": {"model": "yolo11m-pose", "imgsz": 1280, "seed": 1,  "epochs": 250, "patience": 50},
    "m1024": {"model": "yolo11m-pose", "imgsz": 1024, "seed": 42, "epochs": 200, "patience": 40},
}
RUN = ["m1024", "m1280", "l1280"]   # trained in this order, one after another; finished ones are skipped

In [ ]:
# 1. GPU + Google Drive
import os, shutil, subprocess, glob, json, zipfile, hashlib, urllib.request
if not shutil.which("nvidia-smi"):
    raise RuntimeError("NO GPU attached. Runtime > Change runtime type > GPU > Save, then Run all again. "
                       "If Colab says 'Cannot connect to GPU backend', the free GPU time is used up for today.")
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
from google.colab import drive
try:
    drive.mount("/content/drive")
except ValueError as exc:
    raise RuntimeError("Google Drive did not mount. Runtime > Disconnect and delete runtime, Run all again, and in the "
                       "Drive window choose the same Google account and tick EVERY permission box. If no window "
                       "appears, allow pop-ups for colab.research.google.com.") from exc
ROOT = "/content/drive/MyDrive/PerioVision"
EXPORT = f"{ROOT}/export_v3"
for d in ("runs", "current", "export_v3", "eval_v3"):
    os.makedirs(f"{ROOT}/{d}", exist_ok=True)
EVAL = f"{ROOT}/eval_v3"
OLD = f"{ROOT}/current/dental_landmark_yolov8n-pose.pt"
print("deployed model for comparison:", "found" if os.path.exists(OLD) else "NOT FOUND (comparison will be skipped)")

In [ ]:
# 2. Same Ultralytics version as the PerioVision backend
!pip -q install "ultralytics==8.4.21"
import ultralytics; ultralytics.checks()

In [ ]:
# 3. DenPAR (141 MB, checksum-verified)
Z = "/content/denpar.zip"
if not os.path.isdir("/content/denpar/Dataset"):
    urllib.request.urlretrieve("https://zenodo.org/api/records/16645076/files/DenPAR%20Radiographs%20Dataset.zip/content", Z)
    md5 = hashlib.md5(open(Z, "rb").read()).hexdigest()
    assert md5 == "4fcecde6bfa4dc47daec83b1f7856c7f", f"checksum mismatch: {md5}"
    zipfile.ZipFile(Z).extractall("/content/denpar")
for d in ("scripts", "research", "app/ml/landmarks"):
    os.makedirs(f"/content/pv/{d}", exist_ok=True)
for f in ("research/__init__.py", "app/__init__.py", "app/ml/__init__.py", "app/ml/landmarks/__init__.py"):
    open(f"/content/pv/{f}", "w").write("")
print(os.listdir("/content/denpar/Dataset"))

## 4. Code from the repository (written out here so nothing has to be uploaded)

In [ ]:
%%writefile /content/pv/scripts/convert_denpar.py
"""Convert the DenPAR periapical dataset to the YOLO-pose format PerioVision trains on.

DenPAR (Zenodo record 16645076, CC BY 4.0): 1000 periapical radiographs with, per image,
tooth boxes, tooth-wise masks, CEJ points, apex points and alveolar bone-level polylines,
annotated in Labelbox and verified by dental specialists. Points are NOT grouped per tooth,
so this script assigns them:

* each CEJ / apex point goes to the tooth whose mask is nearest (within MAX_DIST px);
* a tooth's apex = mean of its apex points (multi-rooted teeth have several);
* for each CEJ site (mesial / distal) the bone level = the deepest bone-line vertex that
  touches the tooth surface on that side, i.e. where bone meets the root;
* the tooth's keypoints come from its WORST site (largest bone loss), which is what
  periodontal staging uses.

Output per tooth line: 0 cx cy w h  cej_x cej_y v  apex_x apex_y v  crest_x crest_y v
(normalised; v = 2 visible, 0 missing). The official Training / Validation / Testing split is kept.

Usage (from backend/):
    python scripts/convert_denpar.py --src <folder containing Dataset/> --out <output folder>
"""
import argparse
import glob
import json
import os
import shutil

import cv2
import numpy as np

MAX_DIST = 25.0      # px: how far a point may lie from a tooth mask and still belong to it
SPLITS = {"Training": "train", "Validation": "val", "Testing": "test"}


def load_masks(folder):
    masks = []
    for f in sorted(glob.glob(os.path.join(folder, "*.png")), key=lambda p: int("".join(c for c in os.path.basename(p) if c.isdigit()) or 0)):
        m = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        if m is not None and m.max() > 0:
            masks.append(m > 127)
    return masks


def dist_map(mask):
    # distance (px) from every pixel to the nearest mask pixel
    return cv2.distanceTransform((~mask).astype(np.uint8), cv2.DIST_L2, 5)


def at(dm, p):
    h, w = dm.shape
    x, y = int(round(min(max(p[0], 0), w - 1))), int(round(min(max(p[1], 0), h - 1)))
    return float(dm[y, x])


def convert_image(kp, bone, masks):
    teeth = []
    dms = [dist_map(m) for m in masks]
    for m in masks:
        ys, xs = np.nonzero(m)
        teeth.append({"box": [xs.min(), ys.min(), xs.max(), ys.max()], "cx": float(xs.mean()), "cej": [], "apex": []})

    def nearest(p):
        d = [at(dm, p) for dm in dms]
        i = int(np.argmin(d)) if d else -1
        return i if i >= 0 and d[i] <= MAX_DIST else -1

    for name, key in (("CEJ_Points", "cej"), ("Apex_Points", "apex")):
        for p in kp.get(name, []):
            i = nearest(p)
            if i >= 0:
                teeth[i][key].append(np.asarray(p, float))
    vertices = [np.asarray(v, float) for line in bone.get("Bone_Lines", []) for v in line]

    rows = []
    for i, t in enumerate(teeth):
        if not t["cej"] or not t["apex"]:
            continue                      # cannot define the root axis: skip the tooth entirely
        apex = np.mean(t["apex"], axis=0)
        cx_mid = np.mean([c[0] for c in t["cej"]])
        best = None
        for cej in t["cej"]:
            axis = apex - cej
            L2 = float(axis @ axis)
            if L2 < 1:
                continue
            # The crest must come from the SAME side of the tooth as the CEJ. With several CEJ points the side is
            # judged against their midpoint; with a single CEJ, against the tooth's own centre. (Before 2026-10-06 a
            # single CEJ allowed a crest from either side, which paired the CEJ of one side with the crest of the
            # other on 10-14 % of DenPAR teeth.)
            mid = cx_mid if len(t["cej"]) > 1 else t["cx"]
            side = np.sign(cej[0] - mid)
            cands = [v for v in vertices if at(dms[i], v) <= MAX_DIST
                     and (side == 0 or np.sign(v[0] - mid) == side)]
            crest = max(cands, key=lambda v: float((v - cej) @ axis)) if cands else None
            ratio = float((crest - cej) @ axis) / L2 if crest is not None else -1
            if best is None or ratio > best[0]:
                best = (ratio, cej, crest)
        if best is None:
            continue
        rows.append({"box": t["box"], "cej": best[1], "apex": apex, "crest": best[2]})
    return rows


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", required=True, help="folder that contains Dataset/")
    ap.add_argument("--out", required=True)
    args = ap.parse_args()
    root = os.path.join(args.src, "Dataset")
    stats = {}
    for split, short in SPLITS.items():
        os.makedirs(os.path.join(args.out, "images", short), exist_ok=True)
        os.makedirs(os.path.join(args.out, "labels", short), exist_ok=True)
        n_img = n_teeth = n_crest = 0
        for kp_path in sorted(glob.glob(os.path.join(root, split, "Key Points Annotations", "*.json"))):
            stem = os.path.splitext(os.path.basename(kp_path))[0]
            img_path = os.path.join(root, split, "Images", stem + ".jpg")
            bone_path = os.path.join(root, split, "Bone Level Annotations", stem + ".json")
            mask_dir = os.path.join(root, split, "Masks (Tooth-wise)", stem)
            if not (os.path.exists(img_path) and os.path.isdir(mask_dir)):
                continue
            img = cv2.imread(img_path)
            H, W = img.shape[:2]
            kp = json.load(open(kp_path))
            bone = json.load(open(bone_path)) if os.path.exists(bone_path) else {}
            rows = convert_image(kp, bone, load_masks(mask_dir))
            lines = []
            for r in rows:
                x1, y1, x2, y2 = r["box"]
                vals = [0, (x1 + x2) / 2 / W, (y1 + y2) / 2 / H, (x2 - x1) / W, (y2 - y1) / H]
                for p in (r["cej"], r["apex"], r["crest"]):
                    vals += [p[0] / W, p[1] / H, 2] if p is not None else [0, 0, 0]
                lines.append(" ".join(f"{v:.6f}" if isinstance(v, float) else str(v) for v in vals))
                n_crest += r["crest"] is not None
            if not lines:
                continue
            shutil.copy(img_path, os.path.join(args.out, "images", short, stem + ".jpg"))
            open(os.path.join(args.out, "labels", short, stem + ".txt"), "w").write("\n".join(lines) + "\n")
            n_img += 1
            n_teeth += len(lines)
        stats[short] = {"images": n_img, "teeth": n_teeth, "teeth_with_crest": n_crest}
    open(os.path.join(args.out, "pose.yaml"), "w").write(
        f"path: {os.path.abspath(args.out)}\ntrain: images/train\nval: images/val\ntest: images/test\n"
        "kpt_shape: [3, 3]\nflip_idx: [0, 1, 2]\nnames:\n  0: tooth\n")
    print(json.dumps(stats, indent=2))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile /content/pv/scripts/convert_denpar_twosite.py
"""DenPAR -> YOLO-pose with BOTH bone-loss sites per tooth (5 keypoints instead of 3).

Why: the 3-keypoint labels (scripts/convert_denpar.py) keep only each tooth's worst site, which is sometimes
the left and sometimes the right side of the tooth. The model cannot know in advance which side the annotator's
worst site will be, so it learns to put CEJ and crest between the two (the model card's "placed near the middle
of the tooth"). Here every tooth gets both sites, and the worst site is chosen AFTER prediction, by measuring
both. That is how staging is defined anyway.

Keypoints per tooth (image left / right, so a mirror flip just swaps them; flip_idx [2, 3, 0, 1, 4]):
  0 cej_left   1 crest_left   2 cej_right   3 crest_right   4 root_apex
A site with no annotated CEJ or bone line is marked missing (v = 0) and does not contribute to the loss.

Point-to-tooth matching, apex and crest rules are imported unchanged from convert_denpar.py, and the worse of
the two sites equals that script's 3-keypoint label (checked per tooth; mismatches are counted and printed).

Leakage: films that are copies of a film in another split (research.split_audit, docs/DATA_SPLITS.md) are left
out of val / test by default: test 166, 852; val 1028, 158, 899.

Usage (from backend/):
    python scripts/convert_denpar_twosite.py --src <folder containing Dataset/> --out <output folder>
"""
import argparse
import glob
import json
import os
import shutil
import sys

import cv2
import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from convert_denpar import MAX_DIST, SPLITS, at, dist_map, load_masks  # noqa: E402

LEAKED = {"test": {"166", "852"}, "val": {"1028", "158", "899"}}
KPT_NAMES = ("cej_left", "crest_left", "cej_right", "crest_right", "root_apex")


def ratio(cej, crest, apex) -> float:
    axis = apex - cej
    L2 = float(axis @ axis)
    return float((crest - cej) @ axis) / L2 if crest is not None and L2 >= 1 else -1.0


def convert_image(kp, bone, masks):
    """Per tooth: box, apex, a site per side {cej, crest, ratio}, and the 3-keypoint worst site for checking."""
    dms = [dist_map(m) for m in masks]
    teeth = []
    for m in masks:
        ys, xs = np.nonzero(m)
        teeth.append({"box": [xs.min(), ys.min(), xs.max(), ys.max()], "cx": float(xs.mean()), "cej": [], "apex": []})

    def nearest(p):
        d = [at(dm, p) for dm in dms]
        i = int(np.argmin(d)) if d else -1
        return i if i >= 0 and d[i] <= MAX_DIST else -1

    for name, key in (("CEJ_Points", "cej"), ("Apex_Points", "apex")):
        for p in kp.get(name, []):
            i = nearest(p)
            if i >= 0:
                teeth[i][key].append(np.asarray(p, float))
    vertices = [np.asarray(v, float) for line in bone.get("Bone_Lines", []) for v in line]

    rows = []
    for i, t in enumerate(teeth):
        if not t["cej"] or not t["apex"]:
            continue
        apex = np.mean(t["apex"], axis=0)
        cx_mid = np.mean([c[0] for c in t["cej"]])
        sites = {"left": None, "right": None}
        worst = None
        for cej in t["cej"]:
            if float((apex - cej) @ (apex - cej)) < 1:
                continue
            mid = cx_mid if len(t["cej"]) > 1 else t["cx"]                  # same rule as convert_denpar.py
            side = np.sign(cej[0] - mid)
            cands = [v for v in vertices if at(dms[i], v) <= MAX_DIST
                     and (side == 0 or np.sign(v[0] - mid) == side)]
            axis = apex - cej
            crest = max(cands, key=lambda v: float((v - cej) @ axis)) if cands else None
            r = ratio(cej, crest, apex)
            if worst is None or r > worst[0]:
                worst = (r, cej, crest)
            # which side of the tooth: by the CEJ's own x when there are several, else by the tooth centre
            name = "left" if side < 0 else "right"
            if sites[name] is None or r > sites[name]["ratio"]:
                sites[name] = {"cej": cej, "crest": crest, "ratio": r}
        if worst is None:
            continue
        rows.append({"box": t["box"], "apex": apex, "sites": sites, "worst": worst})
    return rows


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", required=True, help="folder that contains Dataset/")
    ap.add_argument("--out", required=True)
    ap.add_argument("--keep-leaked", action="store_true", help="keep the duplicated films in val / test")
    args = ap.parse_args()
    root = os.path.join(args.src, "Dataset")
    stats = {}
    for split, short in SPLITS.items():
        os.makedirs(os.path.join(args.out, "images", short), exist_ok=True)
        os.makedirs(os.path.join(args.out, "labels", short), exist_ok=True)
        n = {"images": 0, "teeth": 0, "both_sites": 0, "one_site": 0, "worst_site_mismatch": 0, "dropped_leaked": 0}
        for kp_path in sorted(glob.glob(os.path.join(root, split, "Key Points Annotations", "*.json"))):
            stem = os.path.splitext(os.path.basename(kp_path))[0]
            if not args.keep_leaked and stem in LEAKED.get(short, ()):
                n["dropped_leaked"] += 1
                continue
            img_path = os.path.join(root, split, "Images", stem + ".jpg")
            bone_path = os.path.join(root, split, "Bone Level Annotations", stem + ".json")
            mask_dir = os.path.join(root, split, "Masks (Tooth-wise)", stem)
            if not (os.path.exists(img_path) and os.path.isdir(mask_dir)):
                continue
            img = cv2.imread(img_path)
            H, W = img.shape[:2]
            kp = json.load(open(kp_path))
            bone = json.load(open(bone_path)) if os.path.exists(bone_path) else {}
            lines = []
            for r in convert_image(kp, bone, load_masks(mask_dir)):
                x1, y1, x2, y2 = r["box"]
                vals = [0, (x1 + x2) / 2 / W, (y1 + y2) / 2 / H, (x2 - x1) / W, (y2 - y1) / H]
                have = 0
                for side in ("left", "right"):
                    s = r["sites"][side]
                    ok = s is not None and s["crest"] is not None
                    have += ok
                    for p in ((s["cej"], s["crest"]) if ok else (None, None)):
                        vals += [p[0] / W, p[1] / H, 2] if p is not None else [0.0, 0.0, 0]
                vals += [r["apex"][0] / W, r["apex"][1] / H, 2]
                best = max((s["ratio"] for s in r["sites"].values() if s is not None and s["crest"] is not None),
                           default=None)
                if best is not None and abs(best - r["worst"][0]) > 1e-9:
                    n["worst_site_mismatch"] += 1
                n["both_sites" if have == 2 else "one_site" if have == 1 else "no_site"] = \
                    n.get("both_sites" if have == 2 else "one_site" if have == 1 else "no_site", 0) + 1
                lines.append(" ".join(f"{v:.6f}" if isinstance(v, float) else str(v) for v in vals))
            if not lines:
                continue
            shutil.copy(img_path, os.path.join(args.out, "images", short, stem + ".jpg"))
            open(os.path.join(args.out, "labels", short, stem + ".txt"), "w").write("\n".join(lines) + "\n")
            n["images"] += 1
            n["teeth"] += len(lines)
        stats[short] = n
    open(os.path.join(args.out, "pose.yaml"), "w").write(
        f"path: {os.path.abspath(args.out)}\ntrain: images/train\nval: images/val\ntest: images/test\n"
        "kpt_shape: [5, 3]\nflip_idx: [2, 3, 0, 1, 4]\nnames:\n  0: tooth\n"
        f"# keypoints: {', '.join(KPT_NAMES)}\n")
    print(json.dumps(stats, indent=2))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile /content/pv/app/ml/landmarks/fusion.py
"""Combine several keypoint readings of the same film into one: mirrored test-time augmentation and model ensembles.

Pure numpy, no app imports, so the Colab evaluation (research/compare_landmark_models.py) runs exactly this code.

A *reading* is one pass of one landmark model over the film: (boxes [n, 4] xyxy, keypoints [n, k, 3] x / y / conf,
box confidences [n]), already mapped back to the original film's coordinates (a mirrored pass is un-mirrored and, for
two-site models, its left / right sites swapped back). The first reading is the anchor: its teeth define the output.
Every other reading contributes the tooth whose box overlaps the anchor tooth best (IoU >= 0.5, each used once).

For each anchor tooth: keypoint coordinates and confidences are the plain mean over the readings that found it (the
same averaging the single-model mirrored reading has always used, so existing calibrations stay valid), and the disagreement is the RANGE (max - min) of the bone-loss values those readings give
(percentage points). With one model and its mirrored pass this range is |normal - mirrored|, the difficulty signal the
adaptive conformal interval and the progression threshold already use; with an ensemble it also captures how much the
models disagree. A tooth found by a single reading gets disagreement None (treated as the hardest case).
"""
from __future__ import annotations

import numpy as np

SITES5 = ((0, 1), (2, 3))          # two-site layout: (cej, crest) on the left, then on the right; apex = 4
SWAP5 = [2, 3, 0, 1, 4]            # left <-> right after a mirror flip


def iou(a, b) -> float:
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def bone_loss_pct(cej, apex, crest) -> float | None:
    """Same formula as app/ml/measurement/bone_loss.py (projection on the root axis, clamped 0-100)."""
    cej, apex, crest = (np.asarray(p, float)[:2] for p in (cej, apex, crest))
    root = apex - cej
    L2 = float(root @ root)
    if L2 <= 1e-6:
        return None
    return max(0.0, min(100.0, float((crest - cej) @ root) / L2 * 100.0))


def reading_pct(k) -> float | None:
    """Bone loss of one tooth in one reading, without any confidence cut-off: 3 keypoints (CEJ, apex, crest) or the
    worse of the two sites for 5 keypoints."""
    if len(k) == 5:
        vals = [bone_loss_pct(k[c], k[4], k[r]) for c, r in SITES5]
        vals = [v for v in vals if v is not None]
        return max(vals) if vals else None
    return bone_loss_pct(k[0], k[1], k[2])


def unmirror(boxes, kpts, width: int):
    """Map a reading of the mirrored film back to the original film (and swap two-site sides)."""
    boxes = np.stack([width - boxes[:, 2], boxes[:, 1], width - boxes[:, 0], boxes[:, 3]], 1) if len(boxes) else boxes
    kpts = kpts.copy()
    if len(kpts):
        kpts[:, :, 0] = width - kpts[:, :, 0]
        if kpts.shape[1] == 5:
            kpts = kpts[:, SWAP5]
    return boxes, kpts


def fuse(readings: list[tuple]) -> list[tuple]:
    """[(boxes, kpts, confs)] -> [(box, fused_kpts, box_conf, disagreement)] for the anchor reading's teeth."""
    if not readings:
        return []
    boxes0, kpts0, confs0 = readings[0]
    out = []
    used = [set() for _ in readings]
    for i in range(len(boxes0)):
        members = [kpts0[i]]
        for r, (boxes, kpts, _c) in enumerate(readings[1:], start=1):
            best_j, best = None, 0.0
            for j in range(len(boxes)):
                if j in used[r]:
                    continue
                v = iou(boxes0[i], boxes[j])
                if v > best:
                    best_j, best = j, v
            if best_j is not None and best >= 0.5:
                used[r].add(best_j)
                members.append(kpts[best_j])
        fused = np.stack(members).mean(0)                           # [k, 3]
        pcts = [p for p in (reading_pct(m) for m in members) if p is not None]
        dis = float(max(pcts) - min(pcts)) if len(members) > 1 and len(pcts) == len(members) else None
        out.append((boxes0[i], fused, float(confs0[i]), dis))
    return out

In [ ]:
%%writefile /content/pv/research/stats.py
"""Small, dependency-light statistics for honest reporting: sample sizes, 95 % confidence intervals,
staging agreement (confusion matrix, exact / within-one-stage accuracy, weighted kappa).

Why each method
  * Proportions with one independent unit per row (films, people): Wilson score interval. It behaves
    well near 0 % / 100 % and for small n, unlike the "p +/- 1.96 SE" textbook interval.
  * Anything measured per TOOTH: teeth in the same film are correlated, so a tooth-level Wilson
    interval would be too narrow. Use the cluster (film-level) bootstrap: resample whole films with
    replacement, recompute the metric, take the 2.5 / 97.5 percentiles.
  * AUC: bootstrap (stratified by class, or by cluster when one film gives two jaws). Hanley & McNeil
    (Radiology 1982;143:29-36) is provided only for when per-film predictions are not available yet,
    and is labelled as an approximation.
  * Weighted kappa (Cohen 1968): linear and quadratic weights over ordered stages I < II < III.

All functions take plain numpy arrays / lists. Nothing here reads files.
"""
from __future__ import annotations

import math
from typing import Callable, Sequence

import numpy as np

Z95 = 1.959963984540054
STAGES = ("I", "II", "III")


# ------------------------------------------------------------------ proportions
def wilson(k: int, n: int, z: float = Z95) -> tuple[float, float]:
    """95 % Wilson score interval for k successes out of n."""
    if n <= 0:
        return (float("nan"), float("nan"))
    p = k / n
    denom = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
    return (max(0.0, centre - half), min(1.0, centre + half))


# ------------------------------------------------------------------ bootstrap
def bootstrap_ci(metric: Callable[[np.ndarray], float], n_items: int, clusters: Sequence | None = None,
                 n_boot: int = 2000, seed: int = 0, alpha: float = 0.05) -> tuple[float, float]:
    """Percentile bootstrap CI for metric(indices).

    `metric` receives an index array into the item list and returns a number. With `clusters` (one
    label per item, e.g. the film each tooth comes from), whole clusters are resampled, which keeps
    within-film correlation and gives an honest (wider) interval."""
    rng = np.random.default_rng(seed)
    if clusters is None:
        groups = [np.array([i]) for i in range(n_items)]
    else:
        labels = np.asarray(clusters)
        _, inv = np.unique(labels, return_inverse=True)
        groups = [np.flatnonzero(inv == g) for g in range(inv.max() + 1)]
    stats = []
    for _ in range(n_boot):
        pick = rng.integers(0, len(groups), len(groups))
        idx = np.concatenate([groups[g] for g in pick])
        v = metric(idx)
        if v is not None and np.isfinite(v):
            stats.append(v)
    if not stats:
        return (float("nan"), float("nan"))
    lo, hi = np.percentile(stats, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return (float(lo), float(hi))


# ------------------------------------------------------------------ AUC
def auc(y: Sequence[int], p: Sequence[float]) -> float:
    """Mann-Whitney AUC with ties counted as one half."""
    y, p = np.asarray(y), np.asarray(p, float)
    pos, neg = p[y == 1], p[y == 0]
    if not len(pos) or not len(neg):
        return float("nan")
    order = np.argsort(np.concatenate([pos, neg]), kind="mergesort")
    allv = np.concatenate([pos, neg])[order]
    ranks = np.empty(len(allv))
    i = 0
    while i < len(allv):                       # average ranks for ties
        j = i
        while j + 1 < len(allv) and allv[j + 1] == allv[i]:
            j += 1
        ranks[i:j + 1] = (i + j) / 2 + 1
        i = j + 1
    r = np.empty(len(allv))
    r[order] = ranks
    return float((r[: len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg)))


def auc_ci_bootstrap(y, p, clusters=None, n_boot: int = 2000, seed: int = 0) -> tuple[float, float]:
    y, p = np.asarray(y), np.asarray(p, float)
    return bootstrap_ci(lambda idx: auc(y[idx], p[idx]), len(y), clusters, n_boot, seed)


def auc_ci_hanley_mcneil(a: float, n_pos: int, n_neg: int, z: float = Z95) -> tuple[float, float]:
    """APPROXIMATE CI from a reported AUC and class counts only. Replace with the bootstrap once
    per-item predictions exist."""
    q1, q2 = a / (2 - a), 2 * a * a / (1 + a)
    se = math.sqrt((a * (1 - a) + (n_pos - 1) * (q1 - a * a) + (n_neg - 1) * (q2 - a * a)) / (n_pos * n_neg))
    return (max(0.0, a - z * se), min(1.0, a + z * se))


# ------------------------------------------------------------------ staging agreement
def stage_of(pct: float) -> str:
    """Same bands as app/ml/measurement/staging.py and scripts/train_panoramic_boneloss.py."""
    return "I" if pct < 15 else "II" if pct <= 33 else "III"


def confusion(ref: Sequence[str], pred: Sequence[str], labels=STAGES) -> np.ndarray:
    """Rows = reference stage, columns = predicted stage."""
    m = np.zeros((len(labels), len(labels)), int)
    pos = {s: i for i, s in enumerate(labels)}
    for r, p in zip(ref, pred):
        m[pos[r], pos[p]] += 1
    return m


def weighted_kappa(ref: Sequence[str], pred: Sequence[str], weights: str = "quadratic", labels=STAGES) -> float:
    """Cohen's weighted kappa. 1 = perfect, 0 = chance-level agreement."""
    m = confusion(ref, pred, labels).astype(float)
    n = m.sum()
    if n == 0:
        return float("nan")
    k = len(labels)
    i, j = np.meshgrid(np.arange(k), np.arange(k), indexing="ij")
    w = np.abs(i - j) / (k - 1) if weights == "linear" else ((i - j) / (k - 1)) ** 2
    expected = np.outer(m.sum(1), m.sum(0)) / n
    denom = (w * expected).sum()
    return float(1 - (w * m).sum() / denom) if denom > 0 else float("nan")


def stage_report(ref_pct, pred_pct, clusters=None, n_boot: int = 2000, seed: int = 0) -> dict:
    """Full staging block for one task: n, confusion matrix, exact / within-one accuracy, kappas, CIs."""
    ref = [stage_of(x) for x in ref_pct]
    pred = [stage_of(x) for x in pred_pct]
    ri = np.array([STAGES.index(s) for s in ref])
    pi = np.array([STAGES.index(s) for s in pred])
    n = len(ref)
    exact = float(np.mean(ri == pi))
    within1 = float(np.mean(np.abs(ri - pi) <= 1))
    ref_a, pred_a = np.array(ref), np.array(pred)

    def ci(fn):
        return bootstrap_ci(fn, n, clusters, n_boot, seed)

    out = {
        "n": n,
        "n_clusters": int(len(set(clusters))) if clusters is not None else n,
        "labels": list(STAGES),
        "confusion_rows_reference_cols_predicted": confusion(ref, pred).tolist(),
        "exact_stage_accuracy": exact,
        "exact_stage_accuracy_ci95": ci(lambda idx: float(np.mean(ri[idx] == pi[idx]))),
        "within_one_stage_accuracy": within1,
        "within_one_stage_accuracy_ci95": ci(lambda idx: float(np.mean(np.abs(ri[idx] - pi[idx]) <= 1))),
        "kappa_linear": weighted_kappa(ref, pred, "linear"),
        "kappa_linear_ci95": ci(lambda idx: weighted_kappa(ref_a[idx], pred_a[idx], "linear")),
        "kappa_quadratic": weighted_kappa(ref, pred, "quadratic"),
        "kappa_quadratic_ci95": ci(lambda idx: weighted_kappa(ref_a[idx], pred_a[idx], "quadratic")),
        "per_reference_stage_n": {s: int(np.sum(ri == k)) for k, s in enumerate(STAGES)},
    }
    if clusters is None:   # independent units: Wilson is exact enough and does not depend on the seed
        out["exact_stage_accuracy_ci95"] = wilson(int(np.sum(ri == pi)), n)
        out["within_one_stage_accuracy_ci95"] = wilson(int(np.sum(np.abs(ri - pi) <= 1)), n)
    return out


def mae_report(ref, pred, clusters=None, n_boot: int = 2000, seed: int = 0) -> dict:
    err = np.abs(np.asarray(pred, float) - np.asarray(ref, float))
    return {
        "n": int(len(err)),
        "mae": float(err.mean()),
        "mae_ci95": bootstrap_ci(lambda idx: float(err[idx].mean()), len(err), clusters, n_boot, seed),
        "median_abs_error": float(np.median(err)),
        "median_abs_error_ci95": bootstrap_ci(lambda idx: float(np.median(err[idx])), len(err), clusters, n_boot, seed),
        "within_10_points": float(np.mean(err <= 10)),
        "within_10_points_ci95": bootstrap_ci(lambda idx: float(np.mean(err[idx] <= 10)), len(err), clusters,
                                              n_boot, seed),
    }


def fmt_ci(value: float, ci: tuple[float, float], pct: bool = False, digits: int = 1) -> str:
    """'73.3 % (95 % CI 68.9-77.2)' or '7.37 (95 % CI 6.8-8.0)'."""
    if value is None or not np.isfinite(value):
        return "n/a"
    s = 100 if pct else 1
    unit = " %" if pct else ""
    return f"{value * s:.{digits}f}{unit} (95% CI {ci[0] * s:.{digits}f}-{ci[1] * s:.{digits}f})"

In [ ]:
%%writefile /content/pv/research/compare_landmark_models.py
"""Head-to-head bone-loss evaluation of landmark models on the SAME films against the SAME reference.

Reference: the 3-keypoint DenPAR labels from scripts/convert_denpar.py (each tooth's worst site), i.e. exactly
what every published PerioVision number is measured against. Models:
  * 3-keypoint model (deployed): bone loss from CEJ / apex / crest, measured when all three confidences >= 0.3
  * 5-keypoint two-site model (scripts/convert_denpar_twosite.py): bone loss at the left and right sites, the
    worse CONFIDENT site is reported. The site confidence cut-off is chosen on the VALIDATION split only.

Both get the app's mirrored test-time augmentation (--tta, on by default): the film is also read mirrored, matched
teeth get averaged keypoints (left / right swapped back for the 5-keypoint model). Matching to reference teeth:
box IoU >= 0.5, greedy, as in scripts/evaluate_landmarks.py.

Usage (Colab, GPU; or CPU for a small --limit smoke test):
  python -m research.compare_landmark_models predict --model old.pt --kpts 3 --images <pose3>/images/val \
      --labels <pose3>/labels/val --out old_val.csv
  python -m research.compare_landmark_models summary --val new_val.csv --test new_test.csv --kpts 5 \
      [--baseline-test old_test.csv] --out compare.json
"""
from __future__ import annotations

import argparse
import csv
import glob
import json
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))  # backend/

import numpy as np  # noqa: E402

from research import stats  # noqa: E402

MIN_KPT_CONF = 0.3                      # app threshold (config/thresholds.json landmarks.min_keypoint_confidence)
SWAP5 = [2, 3, 0, 1, 4]                 # mirror: left <-> right sites


def pct(cej, apex, crest) -> float | None:
    """Identical to app/ml/measurement/bone_loss.py."""
    cej, apex, crest = (np.asarray(p, float) for p in (cej, apex, crest))
    root = apex - cej
    L2 = float(root @ root)
    if L2 <= 1e-6:
        return None
    return max(0.0, min(100.0, float((crest - cej) @ root) / L2 * 100.0))


def iou(a, b) -> float:
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    u = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - ix * iy
    return ix * iy / u if u > 0 else 0.0


def read_ref(path, w, h):
    teeth = []
    for line in open(path, encoding="utf-8"):
        v = list(map(float, line.split()))
        if len(v) < 14:
            continue
        cx, cy, bw, bh = v[1] * w, v[2] * h, v[3] * w, v[4] * h
        kp = [(v[5 + 3 * i] * w, v[6 + 3 * i] * h, v[7 + 3 * i]) for i in range(3)]
        if min(k[2] for k in kp) <= 0:
            continue                                     # no reference bone loss for this tooth
        teeth.append({"bbox": [cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2],
                      "ref": pct(kp[0][:2], kp[1][:2], kp[2][:2])})
    return teeth


def run_model(models, img, kpts: int, tta: bool):
    """[(box, keypoints[k, 3], disagreement or None)]: the app's fusion (app/ml/landmarks/fusion.py) of every model's
    normal and (with tta) mirrored reading. `models` = [(YOLO model, imgsz)]; the first model's normal pass anchors."""
    from app.ml.landmarks import fusion

    def raw(model, imgsz, im):
        r = model.predict(im, imgsz=imgsz, conf=0.05, verbose=False)[0]
        if r.boxes is None or r.keypoints is None or not len(r.boxes):
            return np.zeros((0, 4)), np.zeros((0, kpts, 3)), np.zeros(0)
        return (r.boxes.xyxy.cpu().numpy().copy(), r.keypoints.data.cpu().numpy().copy(),
                r.boxes.conf.cpu().numpy().copy())

    w = img.shape[1]
    readings = []
    for model, imgsz in models:
        readings.append(raw(model, imgsz, img))
        if tta and len(readings[0][0]):
            mb, mk, mc = raw(model, imgsz, np.ascontiguousarray(img[:, ::-1]))
            readings.append((*fusion.unmirror(mb, mk, w), mc))
    if not len(readings[0][0]):
        return []
    return [(box, k, dis) for box, k, _conf, dis in fusion.fuse(readings)]


def tooth_pct(k, kpts: int, site_conf: float) -> float | None:
    """Bone loss of one predicted tooth. 3 keypoints: CEJ, apex, crest. 5: worst confident site."""
    if kpts == 3:
        if min(k[:, 2]) < MIN_KPT_CONF:
            return None
        return pct(k[0, :2], k[1, :2], k[2, :2])
    if k[4, 2] < MIN_KPT_CONF:
        return None
    vals = [pct(k[c, :2], k[4, :2], k[r, :2]) for c, r in ((0, 1), (2, 3))
            if min(k[c, 2], k[r, 2]) >= site_conf]
    vals = [v for v in vals if v is not None]
    return max(vals) if vals else None


def predict(args) -> int:
    import cv2
    from ultralytics import YOLO

    models = []
    for spec in args.model:                       # path or path@imgsz; several = an ensemble
        path, _, size = spec.partition("@")
        m = YOLO(path)
        models.append((m, int(size) if size else (args.imgsz or int(m.overrides.get("imgsz", 1024)))))
    files = sorted(glob.glob(os.path.join(args.images, "*.jpg")) + glob.glob(os.path.join(args.images, "*.png")))
    files = files[: args.limit] if args.limit else files
    rows = []
    for n, f in enumerate(files, 1):
        stem = os.path.splitext(os.path.basename(f))[0]
        lab = os.path.join(args.labels, stem + ".txt")
        img = cv2.imread(f)
        if img is None or not os.path.exists(lab):
            continue
        h, w = img.shape[:2]
        preds, used = run_model(models, img, args.kpts, not args.no_tta), set()
        for t_idx, ref in enumerate(read_ref(lab, w, h)):
            j, best = max(((j, iou(ref["bbox"], p[0])) for j, p in enumerate(preds) if j not in used),
                          key=lambda t: t[1], default=(None, 0.0))
            row = {"image": os.path.basename(f), "tooth_index": t_idx, "ref_pct": round(ref["ref"], 4), "iou": round(best, 4)}
            if j is None or best < 0.5:
                rows.append({**row, "matched": 0})
                continue
            used.add(j)
            k = preds[j][1]
            row.update({"matched": 1, "tta_disagreement": preds[j][2]})
            row.update({f"k{i}_{c}": round(float(k[i, ci]), 3) for i in range(args.kpts) for ci, c in enumerate("xyc")})
            rows.append(row)
        if n % 20 == 0 or n == len(files):
            print(f"[{n}/{len(files)}]", flush=True)
    with open(args.out, "w", newline="", encoding="utf-8") as fh:
        cols = sorted({c for r in rows for c in r}, key=lambda c: (not c.startswith(("image", "tooth", "ref", "iou", "matched")), c))
        wr = csv.DictWriter(fh, fieldnames=cols)
        wr.writeheader()
        wr.writerows(rows)
    print(f"wrote {len(rows)} reference teeth to {args.out}")
    return 0


# ------------------------------------------------------------------ summary
def _load(path, kpts):
    out = []
    for r in csv.DictReader(open(path, encoding="utf-8")):
        k = None
        if r["matched"] == "1":
            k = np.array([[float(r[f"k{i}_{c}"]) for c in "xyc"] for i in range(kpts)])
        out.append({"image": r["image"], "ref": float(r["ref_pct"]), "k": k})
    return out


def score(rows, kpts, site_conf, boot=2000) -> dict:
    found = [r for r in rows if r["k"] is not None]
    meas = [(r, tooth_pct(r["k"], kpts, site_conf)) for r in found]
    meas = [(r, p) for r, p in meas if p is not None]
    ref = np.array([r["ref"] for r, _ in meas])
    pred = np.array([p for _, p in meas])
    films = np.array([r["image"] for r, _ in meas])
    out = {"reference_teeth": len(rows), "matched": len(found), "measured": len(meas),
           "measured_share_of_reference": len(meas) / len(rows) if rows else None}
    out.update(stats.mae_report(ref, pred, films, boot))
    out["staging"] = stats.stage_report(ref, pred, films, boot)
    out["bias_pred_minus_ref"] = float(np.mean(pred - ref))
    return out


def summary(args) -> int:
    val, test = _load(args.val, args.kpts), _load(args.test, args.kpts)
    chosen = None
    if args.kpts == 5:
        # choose the site-confidence cut-off on VALIDATION only: lowest MAE among cut-offs that still measure
        # at least as large a share of teeth as the 3-keypoint rule would need (>= --min-share)
        grid = [round(x, 2) for x in np.arange(0.1, 0.95, 0.05)]
        cands = []
        for c in grid:
            found = [r for r in val if r["k"] is not None]
            m = [(r["ref"], tooth_pct(r["k"], 5, c)) for r in found]
            m = [(a, b) for a, b in m if b is not None]
            if not m:
                continue
            share = len(m) / len(val)
            mae = float(np.mean([abs(a - b) for a, b in m]))
            cands.append({"site_conf": c, "val_mae": mae, "val_share": share})
        ok = [c for c in cands if c["val_share"] >= args.min_share] or cands
        chosen = min(ok, key=lambda c: c["val_mae"])
        print("validation grid:", json.dumps(cands))
    cut = chosen["site_conf"] if chosen else MIN_KPT_CONF
    report = {"kpts": args.kpts, "site_conf_chosen_on_val": chosen, "val": score(val, args.kpts, cut, args.boot),
              "test": score(test, args.kpts, cut, args.boot)}
    if args.baseline_test:
        bk = args.baseline_kpts
        bcut = MIN_KPT_CONF if bk == 3 else args.baseline_site_conf
        base = _load(args.baseline_test, bk)
        report["baseline_test"] = score(base, bk, bcut, args.boot)
        report["baseline_kpts"] = bk
        if bk == 3:
            report["baseline_3kpt_test"] = report["baseline_test"]          # name used by earlier reports
        # paired comparison on teeth both models measured
        nb = {(r["image"], i): r for i, r in enumerate(base)}
        nt = {(r["image"], i): r for i, r in enumerate(test)}
        pairs = []
        for key in nb.keys() & nt.keys():
            a, b = nb[key], nt[key]
            if a["k"] is None or b["k"] is None:
                continue
            pa, pb = tooth_pct(a["k"], bk, bcut), tooth_pct(b["k"], args.kpts, cut)
            if pa is not None and pb is not None:
                pairs.append((key[0], abs(pa - a["ref"]), abs(pb - b["ref"])))
        if pairs:
            films = np.array([p[0] for p in pairs])
            diff = np.array([p[2] - p[1] for p in pairs])
            report["paired_test"] = {
                "teeth": len(pairs), "mae_old": float(np.mean([p[1] for p in pairs])),
                "mae_new": float(np.mean([p[2] for p in pairs])), "mae_new_minus_old": float(diff.mean()),
                "mae_new_minus_old_ci95": stats.bootstrap_ci(lambda i: float(diff[i].mean()), len(diff), films, args.boot),
                "note": "negative = the new model's error is lower; CI excluding 0 = a real difference"}
    json.dump(report, open(args.out, "w", encoding="utf-8"), indent=2, default=float)
    t = report["test"]
    print(f"TEST: measured {t['measured']}/{t['reference_teeth']}, MAE {t['mae']:.2f} {t['mae_ci95']}, "
          f"stage {t['staging']['exact_stage_accuracy']:.3f}")
    if "paired_test" in report:
        p = report["paired_test"]
        print(f"PAIRED: old {p['mae_old']:.2f} vs new {p['mae_new']:.2f}, diff {p['mae_new_minus_old']:.2f} "
              f"CI {p['mae_new_minus_old_ci95']}")
    return 0


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest="cmd", required=True)
    p = sub.add_parser("predict")
    p.add_argument("--model", required=True, action="append",
                   help="model file, optionally path@imgsz; repeat for an ensemble (the first one anchors)")
    p.add_argument("--kpts", type=int, choices=(3, 5), required=True)
    p.add_argument("--images", required=True)
    p.add_argument("--labels", required=True, help="3-keypoint REFERENCE labels (convert_denpar.py)")
    p.add_argument("--out", required=True)
    p.add_argument("--imgsz", type=int)
    p.add_argument("--limit", type=int, default=0)
    p.add_argument("--no-tta", action="store_true")
    s = sub.add_parser("summary")
    s.add_argument("--val", required=True)
    s.add_argument("--test", required=True)
    s.add_argument("--kpts", type=int, choices=(3, 5), required=True)
    s.add_argument("--baseline-test")
    s.add_argument("--baseline-kpts", type=int, choices=(3, 5), default=3)
    s.add_argument("--baseline-site-conf", type=float, default=0.6, help="site cut-off of a 5-keypoint baseline")
    s.add_argument("--min-share", type=float, default=0.95, help="5-kpt: keep at least this share of teeth measured (val)")
    s.add_argument("--boot", type=int, default=2000)
    s.add_argument("--out", required=True)
    args = ap.parse_args(argv)
    return predict(args) if args.cmd == "predict" else summary(args)


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
# 5. Labels from the CORRECTED converter: 3-point reference (what every number is measured against) and 5-point
#    training labels. Films duplicated across splits are left out of the reference too.
if not os.path.exists("/content/pose3/pose.yaml"):
    !cd /content/pv && python scripts/convert_denpar.py --src /content/denpar --out /content/pose3
if not os.path.exists("/content/pose5/pose.yaml"):
    !cd /content/pv && python scripts/convert_denpar_twosite.py --src /content/denpar --out /content/pose5
LEAKED = {"166", "852", "1028", "158", "899"}
for split in ("val", "test"):
    for f in glob.glob(f"/content/pose3/images/{split}/*.jpg"):
        if os.path.splitext(os.path.basename(f))[0] in LEAKED:
            os.remove(f)
print(open("/content/pose5/pose.yaml").read())

In [ ]:
# 6. Train the members in RUN (resumes from Drive; finished members are skipped)
from ultralytics import YOLO
def done(name):
    return os.path.exists(f"{ROOT}/runs/v3_{name}/DONE")
for name in RUN:
    cfg = MEMBERS[name]
    run_dir = f"{ROOT}/runs/v3_{name}"
    if done(name):
        print(name, "already trained"); continue
    print("training", name, cfg)
    if os.path.exists(f"{run_dir}/weights/last.pt"):
        YOLO(f"{run_dir}/weights/last.pt").train(resume=True)
    else:
        YOLO(f"{cfg['model']}.pt").train(
            data="/content/pose5/pose.yaml", imgsz=cfg["imgsz"], epochs=cfg["epochs"], patience=cfg["patience"],
            batch=-1, project=f"{ROOT}/runs", name=f"v3_{name}", exist_ok=True, seed=cfg["seed"],
            fliplr=0.5, flipud=0.5, degrees=5, translate=0.05, scale=0.3, mosaic=1.0, close_mosaic=20,
            hsv_h=0.0, hsv_s=0.0, hsv_v=0.3, cos_lr=True, optimizer="AdamW", lr0=0.001,
            cache="disk", workers=2, plots=True)
    open(f"{run_dir}/DONE", "w").write("ok")
print("finished members:", [n for n in MEMBERS if done(n)])

In [ ]:
# 7. Predictions on validation and test for every finished member, their ensemble, and the deployed model
def run(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, cwd="/content/pv", text=True, capture_output=True)
    print(r.stdout[-2000:], r.stderr[-2000:])
    if r.returncode:
        raise RuntimeError("step failed - see the messages above")
finished = [n for n in MEMBERS if done(n)]
assert finished, "no member has finished training yet: Run all again"
weights = {n: f"{ROOT}/runs/v3_{n}/weights/best.pt" for n in finished}
def predict(tag, specs):
    for split in ("val", "test"):
        out = f"{EVAL}/{tag}_{split}.csv"
        if not os.path.exists(out):
            models = " ".join(f"--model {m}" for m in specs)
            run(f"python -m research.compare_landmark_models predict {models} --kpts 5 "
                f"--images /content/pose3/images/{split} --labels /content/pose3/labels/{split} --out {out}")
for n in finished:
    predict(n, [f"{weights[n]}@{MEMBERS[n]['imgsz']}"])
if os.path.exists(OLD):
    predict("deployed", [f"{OLD}@1024"])

In [ ]:
# 8. Choose on VALIDATION, report on TEST
import numpy as np
def val_mae(tag):
    run(f"python -m research.compare_landmark_models summary --kpts 5 --val {EVAL}/{tag}_val.csv "
        f"--test {EVAL}/{tag}_test.csv --boot 200 --out {EVAL}/{tag}_summary.json")
    r = json.load(open(f"{EVAL}/{tag}_summary.json"))
    return r["val"]["mae"], r["val"]["measured_share_of_reference"]
scores = {n: val_mae(n) for n in finished}
if len(finished) > 1:
    order = sorted(finished, key=lambda n: scores[n][0])          # best member on validation anchors the ensemble
    predict("ensemble", [f"{weights[n]}@{MEMBERS[n]['imgsz']}" for n in order])
    scores["ensemble"] = val_mae("ensemble")
print("validation MAE / share measured:", {k: (round(v[0], 3), round(v[1], 3)) for k, v in scores.items()})
eligible = {k: v for k, v in scores.items() if v[1] >= 0.90} or scores
best = min(eligible, key=lambda k: eligible[k][0])
print("chosen on validation:", best)

base = f"--baseline-test {EVAL}/deployed_test.csv --baseline-kpts 5" if os.path.exists(f"{EVAL}/deployed_test.csv") else ""
run(f"python -m research.compare_landmark_models summary --kpts 5 --val {EVAL}/{best}_val.csv --test {EVAL}/{best}_test.csv "
    f"{base} --out {EXPORT}/v3_comparison.json")
r = json.load(open(f"{EXPORT}/v3_comparison.json"))
t = r["test"]
print(f"CHOSEN ({best}), DenPAR test, corrected reference: MAE {t['mae']:.2f} (95% CI {t['mae_ci95'][0]:.2f}-{t['mae_ci95'][1]:.2f}), "
      f"stage agreement {100 * t['staging']['exact_stage_accuracy']:.1f} %, measured {t['measured']}/{t['reference_teeth']}")
verdict = "no deployed model to compare against"
if "paired_test" in r:
    b, p = r["baseline_test"], r["paired_test"]
    lo, hi = p["mae_new_minus_old_ci95"]
    verdict = "REAL IMPROVEMENT" if hi < 0 else ("REAL WORSENING" if lo > 0 else "NO CLEAR DIFFERENCE")
    print(f"DEPLOYED today, same films and code: MAE {b['mae']:.2f}, stage agreement {100 * b['staging']['exact_stage_accuracy']:.1f} %")
    print(f"Paired on {p['teeth']} teeth: new - deployed = {p['mae_new_minus_old']:+.2f} points (95% CI {lo:+.2f} to {hi:+.2f}) -> {verdict}")

# export the chosen model(s) under the names the app installs
names = ([n for n in sorted(finished, key=lambda n: scores[n][0])] if best == "ensemble" else [best])
for i, n in enumerate(names[:3]):
    target = "dental_landmark_yolov8n-pose.pt" if i == 0 else f"dental_landmark_ens{i + 1}-pose.pt"
    shutil.copy(weights[n], f"{EXPORT}/{target}")
json.dump({"chosen": best, "members": names[:3], "imgsz": {n: MEMBERS[n]["imgsz"] for n in names[:3]},
           "validation": {k: {"mae": v[0], "measured_share": v[1]} for k, v in scores.items()}, "verdict": verdict},
          open(f"{EXPORT}/v3_selection.json", "w"), indent=2)
print("Exported to", EXPORT, ":", sorted(os.listdir(EXPORT)))

## Done
Download `MyDrive/PerioVision/export_v3/` (right-click → Download), unzip it in `Downloads`, and ask for it to be
installed. Installation recalibrates the uncertainty, re-measures every number through the app and re-runs the
progression bench, and keeps the current model if the new one is not better.